# Hands-on LoRA: legal text classification (Overruling)

Here you'll fine-tune a model to answer one yes/no question about a sentence from a court opinion:
**does it overrule a previous case?** ("we expressly overrule it" -> yes; a citation like
"876 f.3d at 1306." -> no).

This is the *classification* cousin of the Qwen persona notebook. The LoRA idea is the same —
freeze the base model, train a tiny adapter — but here we also add a small **classification head**
on top and watch **accuracy jump from ~chance to high**.

- **Model:** `nlpaueb/legal-bert-base-uncased` (a BERT pretrained on legal text).
- **Dataset:** `mteb/OverrulingLegalBenchClassification` (~2,050 short sentences, 2 labels; we make our own train/test split).
- **Runtime:** ~1-3 min on a Colab T4 GPU; a few minutes on CPU (the model is small).

> Every code cell below is heavily commented so you can follow it line by line.

### Run on Google Colab
1. **File > Upload notebook** -> pick this file.
2. (Optional but faster) **Runtime > Change runtime type > T4 GPU**.
3. **Runtime > Run all**.

## 1. Install libraries
Colab already has PyTorch. We add `peft` (LoRA), `datasets`, and `scikit-learn` (for metrics).

In [ ]:
# Each library plays one role (Colab already ships PyTorch):
#   peft         -> the LoRA implementation (LoraConfig, get_peft_model)
#   transformers -> the model, tokenizer, and the Trainer training loop
#   datasets     -> downloads the Overruling data from the Hugging Face Hub
#   accelerate   -> backend the Trainer uses to place tensors on GPU/CPU
#   scikit-learn -> accuracy / F1 metric functions
# -q keeps the install quiet; -U upgrades to recent versions.
%pip install -q -U peft transformers datasets accelerate scikit-learn

## 2. Imports & setup

In [ ]:
import numpy as np
import torch

# From transformers we pull:
#   AutoTokenizer                       -> converts text into token IDs
#   AutoModelForSequenceClassification  -> a base model + a classification "head"
#   TrainingArguments / Trainer         -> the training loop and all its settings
#   DataCollatorWithPadding             -> pads each batch to equal length
#   set_seed                            -> makes the run reproducible
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

set_seed(42)  # fix randomness (weight init, data shuffling) so results are repeatable

# The base model = BERT pretrained on legal text. Changing THIS one string is all it
# takes to try a different backbone, e.g. "distilbert-base-uncased" or "roberta-base".
MODEL = "nlpaueb/legal-bert-base-uncased"

# Use fast 16-bit math only if a GPU is present; on CPU we stay in normal 32-bit.
use_fp16 = torch.cuda.is_available()
print("GPU available:", use_fp16)

# Map between the integer labels in the data and human-readable names. Storing these
# on the model makes its predictions self-describing (it can say "overruling", not "1").
id2label = {0: "not overruling", 1: "overruling"}
label2id = {v: k for k, v in id2label.items()}

## 3. Load and explore the dataset
Each example is a `text` (one sentence) and a `label` (1 = overruling, 0 = not).

**Heads-up on the splits:** as hosted, this dataset follows a LegalBench convention — the `train`
split has only **6 rows** (meant for few-shot prompting), and the real labelled data (~2,048 rows)
lives in `test`. For fine-tuning we take that labelled pool and make our **own** 80/20 train/test split.

In [ ]:
from datasets import load_dataset

# The hosted "train" split is only 6 rows (few-shot prompt examples), while the ~2,048
# real labelled sentences live in "test" — so we load that labelled pool.
full = load_dataset("mteb/OverrulingLegalBenchClassification", split="test")
print("Total labelled sentences:", len(full))

# Cut our OWN 80/20 split from it. seed=42 makes the shuffle reproducible.
split = full.train_test_split(test_size=0.2, seed=42)
train_raw, test_raw = split["train"], split["test"]
print("Our split -> train:", len(train_raw), "| test:", len(test_raw))

# Peek at a few sentences of each class to get a feel for what the model must learn.
print("\n--- Examples labelled OVERRULING (1) ---")
shown = 0
for row in full:
    if row["label"] == 1:
        print(" -", row["text"]); shown += 1
    if shown == 3: break

print("\n--- Examples labelled NOT overruling (0) ---")
shown = 0
for row in full:
    if row["label"] == 0:
        print(" -", row["text"]); shown += 1
    if shown == 3: break

# Check the class balance. If one label dominated, plain accuracy could be misleading
# (that's also why we track F1, not just accuracy).
labels = full["label"]
print("\nClass balance (all): overruling =", sum(labels), "| not =", len(labels) - sum(labels))

## 4. Tokenize
Turn each sentence into token IDs. These sentences are short, so a max length of 128 is plenty.
`DataCollatorWithPadding` pads each batch to its longest example at training time.

In [ ]:
# The tokenizer MUST match the model (it uses the exact vocabulary the model was trained on).
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Convert raw text -> token IDs. truncation + max_length cap over-long inputs; these
# sentences are short, so 128 tokens is more than enough and keeps things fast.
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

# Apply to every row (batched=True processes many at once = faster). We drop the raw
# "text" column so only model-ready fields remain (input_ids, attention_mask,
# token_type_ids, label) — important when using PEFT + Trainer together.
train_tok = train_raw.map(preprocess, batched=True, remove_columns=["text"])
test_tok  = test_raw.map(preprocess, batched=True, remove_columns=["text"])

# Rather than pad every sentence to 128, this collator pads each *batch* to the length
# of its own longest member at training time -> less wasted computation.
collator = DataCollatorWithPadding(tokenizer)
print("Tokenized. Columns:", train_tok.column_names)

## 5. Load the model and attach LoRA
We load Legal-BERT with a fresh 2-class classification head, then wrap it with LoRA.

Two things to notice:
- **`task_type="SEQ_CLS"`** tells LoRA this is a classification model.
- **`modules_to_save=["classifier"]`** — the classification head is brand new and random, so we
  train and save it *in full* (LoRA adapters handle the frozen BERT body underneath).

The printout shows only a small slice of parameters are trainable.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Load Legal-BERT and attach a fresh, randomly-initialised head that outputs 2 logits
# (one score per class). Right now the body is pretrained but the head knows nothing.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label=id2label, label2id=label2id)

# ---- The LoRA configuration: this is the heart of the notebook ----
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,        # this is a sequence-classification model
    r=8,                               # rank = the adapter's capacity (higher = learns more)
    lora_alpha=16,                     # how strongly the adapter is applied (~2x r is common)
    lora_dropout=0.1,                  # dropout inside the adapter, to reduce overfitting
    target_modules=["query", "value"], # which layers get adapters: BERT's attention Q and V
    modules_to_save=["classifier"],    # head is random -> train AND save it in full
)

# get_peft_model freezes the base weights and inserts the trainable LoRA adapters.
model = get_peft_model(model, lora_config)

# Prints something like "trainable params: ~0.9M || all params: ~110M || trainable%: ~0.8".
# That tiny percentage is exactly why LoRA is cheap.
model.print_trainable_parameters()

## 6. Define metrics and the Trainer
We'll score **accuracy** and **F1** on the test set.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Trainer calls this after each evaluation. It hands us the model's raw scores (logits)
# and the true labels; we take the highest-scoring class (argmax) and compute metrics.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds)}

# Every training setting lives in TrainingArguments:
args = TrainingArguments(
    output_dir="overruling-lora",     # folder for any checkpoints/logs
    per_device_train_batch_size=16,   # how many sentences per training step
    per_device_eval_batch_size=32,    # can be larger during eval (no gradients stored)
    num_train_epochs=8,               # how many full passes over the training set
    learning_rate=2e-4,               # LoRA tolerates a higher LR than full fine-tuning
    logging_steps=20,                 # print the training loss every 20 steps
    fp16=use_fp16,                    # 16-bit math on GPU for speed
    report_to="none",                 # don't send logs to W&B / TensorBoard
    save_strategy="no",               # skip writing checkpoints for this quick demo
)

# The Trainer wires together model + data + settings + metrics into one training loop.
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=collator,           # pads batches on the fly (see cell 4)
    compute_metrics=compute_metrics,  # how to score during evaluate()
)

## 7. Score BEFORE training
The classification head starts random, so accuracy should be around chance.

In [ ]:
# Evaluate BEFORE training. The head is random and the LoRA adapters start as no-ops,
# so expect accuracy near chance (~50% on a roughly balanced 2-class task).
# This is our baseline to beat.
before = trainer.evaluate()
print("BEFORE:", {k: round(v, 4) for k, v in before.items() if k in ("eval_accuracy", "eval_f1")})

## 8. Train
A handful of epochs is enough on this small, clean task.

In [ ]:
# The training loop: for each batch it does forward pass -> loss -> backprop, but only
# the LoRA adapters and the classifier head get updated — the BERT body stays frozen.
trainer.train()

## 9. Score AFTER training
Accuracy and F1 should now be high — that gain came from the LoRA adapter + the small head.

In [ ]:
# Evaluate AFTER training on the same test set. Accuracy/F1 should be much higher, and
# that entire improvement lives in the small adapter + head we just trained.
after = trainer.evaluate()
print("AFTER: ", {k: round(v, 4) for k, v in after.items() if k in ("eval_accuracy", "eval_f1")})
print("Accuracy improved:",
      round(before["eval_accuracy"], 3), "->", round(after["eval_accuracy"], 3))

## 10. Try it on your own sentences

In [ ]:
# A helper so you can feed in your own sentences and see the model's prediction.
def predict(sentences):
    model.eval()  # switch off dropout for clean, deterministic inference
    # Tokenize + pad the whole batch, and move the tensors to wherever the model is (GPU/CPU).
    enc = tokenizer(sentences, truncation=True, padding=True, max_length=128,
                    return_tensors="pt").to(model.device)
    with torch.no_grad():                                 # no gradients -> faster, less memory
        preds = model(**enc).logits.argmax(-1).tolist()   # pick the top-scoring class per sentence
    for s, p in zip(sentences, preds):
        print(f"[{id2label[p].upper():>15}]  {s}")

# The first two below should come back "overruling", the last two "not overruling".
predict([
    "We expressly overrule Smith v. Jones and its progeny.",
    "The judgment of the district court is affirmed.",
    "We now hold that our decision in Doe can no longer stand and is overruled.",
    "See id. at 42.",
])

## 11. Save the adapter (it's tiny)

In [ ]:
import os
# Saving a PEFT model writes ONLY the LoRA adapter + the head we trained — not the whole
# base model. That's why this folder is a few MB rather than ~400 MB.
model.save_pretrained("overruling-lora-adapter")

files = os.listdir("overruling-lora-adapter")
size_mb = sum(os.path.getsize(os.path.join("overruling-lora-adapter", f)) for f in files) / 1e6
print("Adapter files:", files)
print(f"Adapter size: {size_mb:.2f} MB  (the head + LoRA weights only)")

## 12. (Reference) Reload the adapter later
```python
from transformers import AutoModelForSequenceClassification
from peft import PeftModel

# 1) load the same base model, 2) apply your saved adapter on top
base  = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label=id2label, label2id=label2id)
tuned = PeftModel.from_pretrained(base, "overruling-lora-adapter")
```

## What to tweak next
- **Lower accuracy than you want?** Raise `num_train_epochs` or `learning_rate`, or `r` (e.g. 16).
- **Swap the base model:** try `distilbert-base-uncased` (lighter) or `roberta-base` (general) and
  compare — you'll usually see the legal-domain model do a bit better on legal text.
- **Harder next task:** move to `unfair_tos` (multi-label) via
  `load_dataset("coastalcph/lex_glue", "unfair_tos")` — it needs a multi-label setup
  (`problem_type="multi_label_classification"` + a sigmoid/threshold), a nice step up.